# Chapter 7: Sequence-to-Sequence and Decoding

**Companion notebook** for *Predicting the Next Words* (Osterrieder, 2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josterri/predicting-the-next-words-notebooks/blob/main/ch07_sequence_to_sequence_and_decoding.ipynb)


## What is in this notebook, and what it needs before it runs

Three cells on how a decoder turns a distribution into a sentence, and on how
anyone decides whether the sentence is any good.

1. **Beam search** over a toy next-token log-probability function, keeping the
   three best partial hypotheses at each step. Pure Python and torch.
2. **Nucleus sampling**, top-p, including the cumulative-probability cutoff.
   Also pure.
3. **BLEU on two example pairs**: a close translation, and a valid paraphrase
   that BLEU scores badly. The second one is the point being made.

The third cell needs `sacrebleu`, which is not installed on the machine this
pack is built on, so no cell below has stored output. A notebook with two cells
filled in and one blank would be worse: you would have to work out which of the
three failed. To run it: `pip install sacrebleu`, then run every cell. Nothing
downloads and nothing takes more than a second.

The edit to make is in cell 1. Set the beam width to 1 and rerun, and beam
search becomes greedy decoding. Then set it to 10 and see whether the sentence
that comes back gets shorter. An unnormalised sum of log probabilities charges
for every extra token, so wider beams tend to prefer shorter outputs, and the
length penalty in the chapter exists for that reason. Watching the problem
appear before you read the fix is worth more than the other way round.


> **This notebook was not executed when it was built, so no cell below has
> stored output.** The reason: cell 3 needs `sacrebleu`, which is not installed on the machine this pack is built on. Cells 1 and 2 need nothing beyond torch.
>
> Nothing here is broken. It is code to read now and to run once you have what
> it needs, and the section above says what that is. Build it yourself with
> `python tools/build_notebook.py ch07` on a machine that has them.


In [ ]:
# Colab does not ship these. Running this cell is a no-op if they are already present.
%pip install -q sacrebleu


In [ ]:
# Seeded before the chapter's own cells run.
#
# The cells below draw on torch. This notebook is committed with its
# output stored, and a stored number that changes on every rebuild is
# noise printed as a result. The bundle originals are read only and
# cannot be fixed where they live, so they are seeded here instead.
#
# Same seed as tools/claim_instances.py, which produced the slide
# numbers, so a number that appears in both places appears once.
import torch
torch.manual_seed(20260729)
print('seeded torch with 20260729')

### 7.3.2 Beam Search


In [ ]:
import torch
import torch.nn.functional as F

def beam_search(log_prob_fn, bos_id, eos_id, beam_width=3, max_len=10):
    """Beam search over a toy next-token log-probability function."""
    # Each beam: (score, token_list)
    beams = [(0.0, [bos_id])]
    completed = []
    for _ in range(max_len):
        candidates = []
        for score, tokens in beams:
            if tokens[-1] == eos_id:
                completed.append((score, tokens))
                continue
            log_probs = log_prob_fn(tokens)          # shape: (vocab_size,)
            topk_lp, topk_ids = log_probs.topk(beam_width)
            for lp, tid in zip(topk_lp.tolist(), topk_ids.tolist()):
                candidates.append((score + lp, tokens + [tid]))
        if not candidates:
            break
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]
    all_hyps = completed + beams
    all_hyps.sort(key=lambda x: x[0] / len(x[1]) ** 0.6, reverse=True)
    return all_hyps[0]  # best length-normalized hypothesis

# --- Toy demo: fixed log-probs for a 6-token vocabulary ---
VOCAB = {0: "SOS", 1: "the", 2: "cat", 3: "sat", 4: "mat", 5: "EOS"}
toy_logits = torch.tensor([[-5., 0.2, -1., -2., -3., -4.],   # after SOS
                           [-5., -3., -0.3, -0.8, -2., -4.],  # after "the"
                           [-5., -3., -3., -1.0, -0.2, -0.5]]) # after "cat"
def toy_log_prob_fn(tokens):
    step = min(len(tokens) - 1, len(toy_logits) - 1)
    return F.log_softmax(toy_logits[step], dim=-1)

best_score, best_tokens = beam_search(toy_log_prob_fn, 0, 5, beam_width=3)
print("Best:", " ".join(VOCAB[t] for t in best_tokens))
print(f"Score: {best_score:.3f}")
# Best: SOS the cat EOS
# Score: -2.166


### 7.3.4 Nucleus (Top-$p$) Sampling


In [ ]:
import torch
import torch.nn.functional as F

def nucleus_sample(logits, p=0.9, temperature=1.0):
    """Sample from the nucleus (top-p) of a logit distribution."""
    logits = logits / temperature
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_idx = probs.sort(descending=True)
    cumulative = sorted_probs.cumsum(dim=-1)
    # Find cutoff: first index where cumulative prob >= p
    mask = cumulative - sorted_probs >= p   # exclude tokens beyond nucleus
    sorted_probs[mask] = 0.0
    sorted_probs /= sorted_probs.sum()      # renormalize
    # Sample from the truncated distribution
    sample_idx = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_idx[sample_idx].item()

# --- Demo: sample 8 tokens from a synthetic 50-token distribution ---
torch.manual_seed(42)
logits = torch.randn(50)   # random logits for 50 tokens
print("p=0.5:", [nucleus_sample(logits, p=0.5) for _ in range(8)])
print("p=0.9:", [nucleus_sample(logits, p=0.9) for _ in range(8)])
print("p=0.95:", [nucleus_sample(logits, p=0.95) for _ in range(8)])
# p=0.5:  [13, 13, 13, 45, 13, 45, 13, 13]   <- tight nucleus, low diversity
# p=0.9:  [13, 45, 30, 13, 2, 45, 13, 30]     <- moderate diversity
# p=0.95: [13, 45, 30, 2, 17, 13, 45, 6]      <- wider nucleus, more diversity


### 7.4.1 BLEU Score


In [ ]:
import sacrebleu

# Example 1: close translation
refs = [["The cat is on the mat."]]
cand = ["The cat on the mat."]
bleu = sacrebleu.corpus_bleu(cand, refs)
print(f"Example 1 — BLEU: {bleu.score:.1f}")
print(f"  Precisions: {[f'{p:.1f}' for p in bleu.precisions]}")
print(f"  BP: {bleu.bp:.3f}, ratio: {bleu.sys_len}/{bleu.ref_len}")
# Example 1 — BLEU: 51.2
#   Precisions: ['100.0', '80.0', '50.0', '33.3']
#   BP: 0.846, ratio: 6/7

# Example 2: valid paraphrase, low BLEU
refs2 = [["The cat is on the mat."]]
cand2 = ["A feline rests upon the rug."]
bleu2 = sacrebleu.corpus_bleu(cand2, refs2)
print(f"\nExample 2 — BLEU: {bleu2.score:.1f}")
print(f"  Precisions: {[f'{p:.1f}' for p in bleu2.precisions]}")
# Example 2 — BLEU: 7.8
#   Precisions: ['28.6', '8.3', '5.0', '3.1']
# A perfectly valid translation scores near zero — BLEU's key limitation.


---

## Summary

This notebook demonstrated the key code examples from Chapter 7: Sequence-to-Sequence and Decoding. For the full mathematical exposition and discussion, refer to the textbook chapter.
